In [1]:
"""
Week 5 — Additional Models (Decision Tree, Random Forest)

Goal: extend the leak-safe Week 4 pipeline (chronological split, CV-safe
City target encoding, Pipeline/ColumnTransformer fit on train only) to two
additional regressors, and compare their test-month R² against the Linear
Regression baseline on identical train/validation/test windows.

Design choices:

1. Everything about the leakage-safe machinery from 03_baseline_model.py is
   reused unchanged: outlier thresholds learned on train only (Sec. 03),
   K-fold CV-safe target encoding for City (Sec. 05), and the
   Pipeline/ColumnTransformer fit exclusively on the training window
   (Sec. 07). Only the final estimator (the "regressor" step) changes
   between models — everything upstream of it is identical, so any R²
   difference between models reflects the estimator, not a difference in
   preprocessing.

2. The training-window length X is tuned independently for each model on
   the validation month (Sec. 01: never tune on the test month), since a
   tree ensemble may prefer a different amount of training history than a
   linear model.

3. Train-set R² is also reported alongside test R² for every model, purely
   as an overfitting diagnostic (a large train/test gap is a classic
   symptom of an unconstrained Decision Tree memorizing training rows).
   This is not a formal metric from the best-practices doc — it exists
   only to document model behavior, per this week's deliverable.

4. Random seeds are fixed throughout for reproducibility (Sec. 11).
"""

'\nWeek 5 — Additional Models (Decision Tree, Random Forest)\n\nGoal: extend the leak-safe Week 4 pipeline (chronological split, CV-safe\nCity target encoding, Pipeline/ColumnTransformer fit on train only) to two\nadditional regressors, and compare their test-month R² against the Linear\nRegression baseline on identical train/validation/test windows.\n\nDesign choices:\n\n1. Everything about the leakage-safe machinery from 03_baseline_model.py is\n   reused unchanged: outlier thresholds learned on train only (Sec. 03),\n   K-fold CV-safe target encoding for City (Sec. 05), and the\n   Pipeline/ColumnTransformer fit exclusively on the training window\n   (Sec. 07). Only the final estimator (the "regressor" step) changes\n   between models — everything upstream of it is identical, so any R²\n   difference between models reflects the estimator, not a difference in\n   preprocessing.\n\n2. The training-window length X is tuned independently for each model on\n   the validation month (Sec. 

In [2]:
import numpy as np
import pandas as pd
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [4]:
TARGET = "ClosePrice"
HIGH_CARD_CAT = ["City"]
LOW_CARD_CAT = [
    "PropertySubType",
    "Levels",
    "AttachedGarageYN",
    "PoolPrivateYN",
    "FireplaceYN",
    "NewConstructionYN",
]

In [5]:
# ---------------------------------------------------------------------------
# Load cleaned data (output of 02_preprocessing.py)
# ---------------------------------------------------------------------------
df = pd.read_csv("cleaned_single_family_sales_preprocessed.csv", parse_dates=["CloseDate"])

In [6]:
# Defensive re-normalization (same rationale as 03_baseline_model.py): guards
# against bool/str mixed-type columns re-appearing if this CSV is ever
# regenerated or hand-edited.
for col in HIGH_CARD_CAT + LOW_CARD_CAT:
    df[col] = df[col].where(df[col].isna(), df[col].astype(str))

In [7]:
NUMERIC_COLS = [
    c for c in df.columns
    if c not in [TARGET, "CloseDate"] + HIGH_CARD_CAT + LOW_CARD_CAT
]

In [8]:
def month_period(series):
    return series.dt.to_period("M")


def get_window(data, end_month_exclusive, n_months):
    """Rows with CloseDate in [end_month_exclusive - n_months, end_month_exclusive)."""
    start = end_month_exclusive - n_months
    periods = month_period(data["CloseDate"])
    return data[(periods >= start) & (periods < end_month_exclusive)]


def get_month(data, month):
    return data[month_period(data["CloseDate"]) == month]

In [9]:
def kfold_target_encode(train_series, train_target, apply_series_list,
                         n_splits=5, smoothing=20, seed=RANDOM_SEED):
    """CV-safe target encoding (Best Practices Sec. 05): out-of-fold means
    for the training rows themselves (so a row's own label never leaks into
    its own encoding), and a single full-training-set mapping applied to
    validation/test rows. Identical to the Week 4 implementation."""
    train_series = train_series.reset_index(drop=True)
    train_target = train_target.reset_index(drop=True)
    global_mean = train_target.mean()

    encoded_train = np.zeros(len(train_series))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fit_idx, hold_idx in kf.split(train_series):
        fit_vals, fit_target = train_series.iloc[fit_idx], train_target.iloc[fit_idx]
        stats = fit_target.groupby(fit_vals).agg(["mean", "count"])
        smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
        encoded_train[hold_idx] = train_series.iloc[hold_idx].map(smoothed).fillna(global_mean).to_numpy()

    full_stats = train_target.groupby(train_series).agg(["mean", "count"])
    full_smoothed = (full_stats["mean"] * full_stats["count"] + global_mean * smoothing) / (full_stats["count"] + smoothing)
    encoded_apply = [s.map(full_smoothed).fillna(global_mean).to_numpy() for s in apply_series_list]
    return encoded_train, encoded_apply

In [10]:
def compute_metrics(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred)}

In [11]:
def build_pipeline(regressor):
    """Same leak-safe ColumnTransformer as Week 4 (Sec. 07): imputers /
    scaler / one-hot encoder are all fit on the training split only. Only
    the final regressor step is swapped between models, so preprocessing
    never differs between Linear Regression / Decision Tree / Random Forest.
    """
    preprocessor = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUMERIC_COLS + ["City_target_enc"]),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", min_frequency=10)),
        ]), LOW_CARD_CAT),
    ])
    return Pipeline([
        ("preprocess", preprocessor),
        ("regressor", regressor),
    ])

In [12]:
def fit_and_evaluate(train_df, eval_df, regressor):
    """Fits the full leak-safe pipeline (with the given regressor) on
    train_df only, then scores it on both train_df and eval_df. Returns
    train R² alongside eval R² purely as an overfitting diagnostic — not a
    formal accuracy metric — so tree-based model behavior can be documented.
    """
    train_df = train_df.copy()
    eval_df = eval_df.copy()

    # Outlier thresholds learned from TRAINING data only (Sec. 03), then the
    # identical frozen cutoffs are applied to the evaluation set.
    lo, hi = train_df[TARGET].quantile([0.005, 0.995])
    train_df = train_df[(train_df[TARGET] >= lo) & (train_df[TARGET] <= hi)]
    eval_df = eval_df[(eval_df[TARGET] >= lo) & (eval_df[TARGET] <= hi)]

    y_train = train_df[TARGET].reset_index(drop=True)
    y_eval = eval_df[TARGET].reset_index(drop=True)

    city_train_enc, (city_eval_enc,) = kfold_target_encode(
        train_df["City"], y_train, [eval_df["City"]]
    )

    X_train = train_df.drop(columns=[TARGET, "CloseDate", "City"]).reset_index(drop=True)
    X_train["City_target_enc"] = city_train_enc
    X_eval = eval_df.drop(columns=[TARGET, "CloseDate", "City"]).reset_index(drop=True)
    X_eval["City_target_enc"] = city_eval_enc

    model = build_pipeline(regressor)

    # Suppress the expected, understood "unknown category at predict time"
    # warning (a rare LOW_CARD_CAT value appearing for the first time in the
    # eval window is normal, not a bug — handle_unknown="ignore" encodes it
    # as all-zeros). Any other warning still surfaces.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Found unknown categories.*",
            category=UserWarning,
        )
        model.fit(X_train, y_train)
        y_pred_eval = model.predict(X_eval)
        y_pred_train = model.predict(X_train)

    eval_metrics = compute_metrics(y_eval, y_pred_eval)
    train_metrics = compute_metrics(y_train, y_pred_train)
    return model, train_metrics, eval_metrics, y_eval, y_pred_eval

In [13]:
# ---------------------------------------------------------------------------
# Chronological split (Sec. 01): identical to Week 4 — test = last complete
# month, validation = second-to-last month.
# ---------------------------------------------------------------------------
months_sorted = sorted(month_period(df["CloseDate"]).unique())
if len(months_sorted) < 3:
    raise ValueError("Need at least 3 distinct months for a train/validation/test split.")

test_month = months_sorted[-1]
val_month = months_sorted[-2]
print(f"Validation month: {val_month}   Test month: {test_month}")

Validation month: 2026-04   Test month: 2026-05


In [14]:
# ---------------------------------------------------------------------------
# Candidate models. Tree-based estimators are given light, explicit
# regularization (max_depth / min_samples_leaf) rather than left to grow
# unbounded — an unconstrained Decision Tree on ~10-20k training rows will
# otherwise memorize the training set (near-perfect train R², much weaker
# test R²). Random Forest bags many such trees together, which is expected
# to reduce that variance relative to a single tree.
# ---------------------------------------------------------------------------
def make_models():
    return {
        "Linear Regression (baseline)": LinearRegression(),
        "Decision Tree": DecisionTreeRegressor(
            max_depth=10,
            min_samples_leaf=20,
            random_state=RANDOM_SEED,
        ),
        "Random Forest": RandomForestRegressor(
            n_estimators=300,
            max_depth=14,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=RANDOM_SEED,
        ),
    }

candidate_X = [3, 6, 9, 12, 18, 24]

In [15]:
# ---------------------------------------------------------------------------
# Tune the training-window length X separately for each model on the
# VALIDATION month only (Sec. 01: never tune on the test month). A tree
# ensemble may prefer a different amount of history than a linear model, so
# X is not assumed to carry over from the Week 4 baseline.
# ---------------------------------------------------------------------------
best_X_per_model = {}
val_results_per_model = {}

for name, regressor in make_models().items():
    print(f"\nTuning training-window length X for: {name}")
    val_results = {}
    for x in candidate_X:
        train_window = get_window(df, val_month, x)
        val_set = get_month(df, val_month)
        if len(train_window) < 100 or len(val_set) < 20:
            print(f"  X={x:>2} months -> skipped (not enough data)")
            continue
        _, _, val_metrics, _, _ = fit_and_evaluate(train_window, val_set, regressor)
        val_results[x] = val_metrics
        print(f"  X={x:>2} months -> val R2={val_metrics['R2']:.4f}")
    if not val_results:
        raise ValueError(f"No candidate training window had enough data for {name}.")
    best_X = max(val_results, key=lambda x: val_results[x]["R2"])
    best_X_per_model[name] = best_X
    val_results_per_model[name] = val_results
    print(f"  Selected X = {best_X} months for {name}")


Tuning training-window length X for: Linear Regression (baseline)
  X= 3 months -> val R2=0.7263
  X= 6 months -> val R2=0.7528
  X= 9 months -> val R2=0.7607
  X=12 months -> val R2=0.7663
  X=18 months -> val R2=0.7689
  X=24 months -> val R2=0.7705
  Selected X = 24 months for Linear Regression (baseline)

Tuning training-window length X for: Decision Tree
  X= 3 months -> val R2=0.7474
  X= 6 months -> val R2=0.7912
  X= 9 months -> val R2=0.8087
  X=12 months -> val R2=0.8051
  X=18 months -> val R2=0.8057
  X=24 months -> val R2=0.8151
  Selected X = 24 months for Decision Tree

Tuning training-window length X for: Random Forest
  X= 3 months -> val R2=0.7888
  X= 6 months -> val R2=0.8186
  X= 9 months -> val R2=0.8336
  X=12 months -> val R2=0.8359
  X=18 months -> val R2=0.8381
  X=24 months -> val R2=0.8426
  Selected X = 24 months for Random Forest


In [16]:
# ---------------------------------------------------------------------------
# Final comparison: retrain each model with its own selected X on the window
# ending right before the test month, then touch the test month exactly
# once per model.
# ---------------------------------------------------------------------------
comparison_rows = []
fitted_models = {}
predictions = {}

for name, regressor in make_models().items():
    best_X = best_X_per_model[name]
    final_train = get_window(df, test_month, best_X)
    test_set = get_month(df, test_month)
    model, train_metrics, test_metrics, y_test_actual, y_test_pred = fit_and_evaluate(
        final_train, test_set, regressor
    )
    fitted_models[name] = model
    predictions[name] = (y_test_actual, y_test_pred)
    comparison_rows.append({
        "Model": name,
        "Training window X (months)": best_X,
        "Train R2": round(train_metrics["R2"], 4),
        "Test R2": round(test_metrics["R2"], 4),
        "Train-Test R2 gap": round(train_metrics["R2"] - test_metrics["R2"], 4),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("Test R2", ascending=False).reset_index(drop=True)

In [17]:
print("=== Model Comparison: Test-Month R² (test month touched once per model) ===\n")
print(comparison_df.to_string(index=False))

=== Model Comparison: Test-Month R² (test month touched once per model) ===

                       Model  Training window X (months)  Train R2  Test R2  Train-Test R2 gap
               Random Forest                          24    0.8955   0.8395             0.0560
               Decision Tree                          24    0.8367   0.8127             0.0241
Linear Regression (baseline)                          24    0.7674   0.7671             0.0003


In [18]:
# ---------------------------------------------------------------------------
# Improvement over baseline
# ---------------------------------------------------------------------------
baseline_r2 = comparison_df.loc[
    comparison_df["Model"] == "Linear Regression (baseline)", "Test R2"
].iloc[0]

print("\n=== Improvement over Linear Regression baseline ===")
for _, row in comparison_df.iterrows():
    if row["Model"] == "Linear Regression (baseline)":
        continue
    delta = row["Test R2"] - baseline_r2
    print(f"{row['Model']:<28} Test R2={row['Test R2']:.4f}  "
          f"(Δ vs baseline: {delta:+.4f})")


=== Improvement over Linear Regression baseline ===
Random Forest                Test R2=0.8395  (Δ vs baseline: +0.0724)
Decision Tree                Test R2=0.8127  (Δ vs baseline: +0.0456)


In [19]:
# ---------------------------------------------------------------------------
# Feature importance for the tree-based models (Decision Tree, Random
# Forest) — helps document *why* a model over/under-performs, not just the
# headline R2. Linear Regression has no equivalent single-number importance
# (its information is in signed, scaled coefficients), so it is skipped here.
# ---------------------------------------------------------------------------
for name in ("Decision Tree", "Random Forest"):
    model = fitted_models[name]
    feature_names = model.named_steps["preprocess"].get_feature_names_out()
    importances = model.named_steps["regressor"].feature_importances_
    top = (
        pd.Series(importances, index=feature_names)
        .sort_values(ascending=False)
        .head(10)
    )
    print(f"\nTop 10 features — {name}:")
    print(top.to_string())


Top 10 features — Decision Tree:
num__City_target_enc          0.630138
num__LivingArea               0.335134
num__YearBuilt                0.011882
cat__PoolPrivateYN_Unknown    0.008172
num__LotSize                  0.007297
num__Bathrooms                0.001809
num__LivingArea_missing       0.000969
cat__Levels_Two               0.000895
num__Bedrooms                 0.000751
cat__FireplaceYN_True         0.000671

Top 10 features — Random Forest:
num__City_target_enc          0.606156
num__LivingArea               0.318873
num__YearBuilt                0.021978
num__LotSize                  0.020130
cat__PoolPrivateYN_Unknown    0.007104
num__Bathrooms                0.005679
num__ParkingTotal             0.003605
num__Bedrooms                 0.003351
num__GarageSpaces             0.002058
cat__Levels_Two               0.001522


In [20]:
"""
Documented model behavior (strengths / weaknesses)
----------------------------------------------------
Read alongside the printed comparison table and train/test R2 gaps above.

Linear Regression (baseline)
  + Fast to fit, fully interpretable coefficients, smallest train-test gap
    (it cannot memorize training rows, so it has little room to overfit).
  - Assumes an additive, linear relationship between features and price.
    Real-estate pricing has interaction effects (e.g. an extra bedroom is
    worth more in a large house than a small one) and nonlinear effects
    (diminishing returns to lot size) that a linear model cannot represent
    without manual feature engineering.

Decision Tree
  + Captures nonlinearities and interactions automatically; splits are
    human-readable (can literally read off the decision path for a home).
  - A single tree is high-variance: small changes in the training window
    can produce a very different tree. Expect the largest train-test R2
    gap of the three models here even with max_depth/min_samples_leaf
    regularization — that gap is itself the diagnostic that this model is
    partly memorizing training-window idiosyncrasies rather than learning
    structure that generalizes to the test month.

Random Forest
  + Averaging many decorrelated trees (bagging + random feature subsets)
    reduces the variance problem that hurts a single Decision Tree, while
    keeping the ability to model nonlinearities/interactions. Expect this
    to have the smallest train-test gap of the two tree-based models, and
    typically the strongest test R2 of all three.
  - Slower to train/predict than either alternative, and loses the direct
    interpretability of a single tree or a linear coefficient (feature
    importances above are a partial substitute, but do not show direction
    of effect the way a linear coefficient does).
  - Still trained per training-window here, and per Sec. 01 the reported
    test R2 is a single cutoff; the rolling-origin backtest performed for
    the baseline in Week 4 has not been re-run for these two models in this
    script and would be a natural next step before treating any single
    model's test R2 as a stable estimate of production performance.
"""

"\nDocumented model behavior (strengths / weaknesses)\n----------------------------------------------------\nRead alongside the printed comparison table and train/test R2 gaps above.\n\nLinear Regression (baseline)\n  + Fast to fit, fully interpretable coefficients, smallest train-test gap\n    (it cannot memorize training rows, so it has little room to overfit).\n  - Assumes an additive, linear relationship between features and price.\n    Real-estate pricing has interaction effects (e.g. an extra bedroom is\n    worth more in a large house than a small one) and nonlinear effects\n    (diminishing returns to lot size) that a linear model cannot represent\n    without manual feature engineering.\n\nDecision Tree\n  + Captures nonlinearities and interactions automatically; splits are\n    human-readable (can literally read off the decision path for a home).\n  - A single tree is high-variance: small changes in the training window\n    can produce a very different tree. Expect the larges

In [21]:
# ---------------------------------------------------------------------------
# Save comparison table and each model's test-set predictions
# ---------------------------------------------------------------------------
comparison_df.to_csv("model_comparison.csv", index=False)

all_predictions = []
for name, (y_actual, y_pred) in predictions.items():
    all_predictions.append(pd.DataFrame({
        "Model": name,
        "ActualPrice": y_actual,
        "PredictedPrice": y_pred,
    }))
pd.concat(all_predictions, ignore_index=True).to_csv("model_comparison_predictions.csv", index=False)

print("\nSaved model_comparison.csv and model_comparison_predictions.csv")


Saved model_comparison.csv and model_comparison_predictions.csv
